# Qwen3.5-9B LoRA+ 글쓰기 채점 모델

과제기술서의 공식 프롬프트, 정수 출력, temperature=0/seed=42, 12항목 rationale Judge와 Hugging Face URL 제출 규정을 반영한 실행 노트북입니다. Linux/Python 3.11 및 L40S/A100 48GB 이상에서 실행하세요.

In [ ]:
# 새 GPU 환경에서 한 번만 실행
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
from pathlib import Path
import json

from essay_scorer.config import RunConfig
from essay_scorer.data import load_examples, load_rationale_records, validate_rationale_coverage
from essay_scorer.evaluation import integer_oracle_metrics

cfg = RunConfig()
SELECTED_LORAPLUS_RATIO = 16.0  # ratio 8/16 파일럿 평가 후 변경
train = load_examples(cfg.train_path)
validation = load_examples(cfg.validation_path)
print({'train': len(train), 'validation': len(validation)})
print(json.dumps(integer_oracle_metrics(validation), ensure_ascii=False, indent=2))

## 1. rationale 자기증류

별도 터미널에서 다음 로컬 서버를 먼저 실행합니다. 외부 API는 허용되지 않습니다.

```bash
vllm serve Qwen/Qwen3.5-9B --language-model-only --max-model-len 32768 --reasoning-parser qwen3
```

In [ ]:
from essay_scorer.generation import LocalOpenAIChatGenerator
from essay_scorer.rationales import RationaleBuilder, generate_rationale_file

RUN_RATIONALE_GENERATION = False
if RUN_RATIONALE_GENERATION:
    generator = LocalOpenAIChatGenerator('http://127.0.0.1:8000/v1', cfg.base_model_id)
    builder = RationaleBuilder(
        generator,
        max_attempts=cfg.rationale_max_attempts,
        minimum_item_score=cfg.rationale_min_item_score,
        minimum_mean_score=cfg.rationale_min_mean_score,
    )
    print(generate_rationale_file(train, builder, 'artifacts/train_rationales.jsonl'))
    print(generate_rationale_file(validation, builder, 'artifacts/validation_rationales.jsonl'))

## 2. 검증된 저·중·고 3-shot 선정

In [ ]:
from essay_scorer.fewshot import select_balanced_demos, save_demos

train_rationales_path = Path('artifacts/train_rationales.jsonl')
if train_rationales_path.exists():
    train_rationales = load_rationale_records(train_rationales_path)
    print(validate_rationale_coverage(train, train_rationales, cfg.min_rationale_keep_rate))
    demos = select_balanced_demos(train, train_rationales)
    save_demos('artifacts/demos.json', demos)
    print(json.dumps(demos, ensure_ascii=False, indent=2))

선택적으로 base 모델에서 6가지 few-shot 순서를 validation 전체로 비교합니다. 2,400회 추론이 필요하므로 별도 GPU 세션에서 실행하세요.

In [ ]:
from essay_scorer.evaluation import choose_demo_order
from essay_scorer.fewshot import all_demo_orders, load_demos
from essay_scorer.generation import TransformersChatGenerator

RUN_DEMO_ORDER_SEARCH = False
if RUN_DEMO_ORDER_SEARCH:
    demos = load_demos('artifacts/demos.json')
    base_generator = TransformersChatGenerator.from_pretrained(cfg.base_model_id)
    ordered, reports = choose_demo_order(base_generator, validation, all_demo_orders(demos))
    save_demos('artifacts/demos_ordered.json', ordered)
    Path('artifacts/demo_order_reports.json').write_text(
        json.dumps(reports, ensure_ascii=False, indent=2), encoding='utf-8'
    )

## 3. LoRA+ 파일럿과 본 학습

ratio 8과 16을 각각 1 epoch 실행해 검증한 뒤 선택한 비율로 3 epoch를 학습합니다. 아래 직접 호출은 단일 GPU용이며 분산 실행이 필요하면 README의 `accelerate launch` 명령을 사용합니다.

In [ ]:
from essay_scorer.training import train_loraplus

RUN_TRAINING = False
if RUN_TRAINING:
    demos_path = Path('artifacts/demos_ordered.json')
    if not demos_path.exists():
        demos_path = Path('artifacts/demos.json')
    demos = load_demos(demos_path)
    train_rationales = load_rationale_records('artifacts/train_rationales.jsonl')
    train_cfg = RunConfig(output_dir=Path('artifacts/main'), loraplus_lr_ratio=SELECTED_LORAPLUS_RATIO, epochs=3)
    manifest = train_loraplus(train_cfg, train, train_rationales, demos)
    print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 4. 공식 조건 validation 평가

In [ ]:
from essay_scorer.evaluation import evaluate_generator, load_adapter_generator

RUN_EVALUATION = False
if RUN_EVALUATION:
    checkpoint = Path('artifacts/main/epoch-3')
    adapter_generator = load_adapter_generator(cfg.base_model_id, checkpoint)
    _, metrics = evaluate_generator(
        adapter_generator,
        validation,
        output_path='artifacts/validation_predictions.jsonl',
        retries=2,
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 5. train+validation 최종 재학습 및 병합

In [ ]:
RUN_FINAL_REFIT = False
if RUN_FINAL_REFIT:
    final_demos_path = Path('artifacts/demos_ordered.json')
    if not final_demos_path.exists():
        final_demos_path = Path('artifacts/demos.json')
    demos = load_demos(final_demos_path)
    all_examples = train + validation
    all_rationales = (
        load_rationale_records('artifacts/train_rationales.jsonl')
        + load_rationale_records('artifacts/validation_rationales.jsonl')
    )
    final_cfg = RunConfig(output_dir=Path('artifacts/final'), loraplus_lr_ratio=SELECTED_LORAPLUS_RATIO, epochs=3)
    train_loraplus(final_cfg, all_examples, all_rationales, demos)

In [ ]:
from essay_scorer.export import merge_adapter, write_model_card

RUN_MERGE = False
if RUN_MERGE:
    merge_adapter(cfg.base_model_id, 'artifacts/final/epoch-3', 'artifacts/merged_model')
    write_model_card('artifacts/merged_model', base_model_id=cfg.base_model_id)

## 6. Hugging Face 공개 업로드

`HF_TOKEN`은 환경변수로만 설정합니다. 저장소는 public·ungated로 생성됩니다.

In [ ]:
import os
from essay_scorer.export import upload_public_model, vllm_docker_command

HF_REPO_ID = os.environ.get('HF_REPO_ID', '')  # 예: organization/model-name
RUN_UPLOAD = False
if RUN_UPLOAD:
    assert HF_REPO_ID and os.environ.get('HF_TOKEN')
    print(upload_public_model('artifacts/merged_model', HF_REPO_ID))
    print(vllm_docker_command(HF_REPO_ID))